# 02b: ...

### Импорты и пути

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedGroupKFold

DATA_DIR = Path("../data")
FINAL_DIR = DATA_DIR / "physical_signs_final"

In [2]:
koi = pd.read_csv(FINAL_DIR / "koi_clean.csv")

y = koi["label"].astype(int)
groups = koi["kepid"]
id_cols = ["label", "kepid", "kepoi_name"]
X = koi.drop(columns = id_cols)

### Сплит по группам

In [3]:
sgkf_holdout = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=1)
train_idx, test_idx = next(sgkf_holdout.split(X, y, groups=groups))

### Диагностика

In [4]:
train_kepids = set(groups.iloc[train_idx])
test_kepids = set(groups.iloc[test_idx])
overlap = train_kepids & test_kepids

print(f"Пересечение kepid между train и test: {len(overlap)}")
assert len(overlap) == 0, f"Утечка! Пересекаются kepid: {list(overlap)[:5]}"
print("Группы не пересекаются")

print(f"\nБаланс классов:")
print(f"  Train: 1 = {y.iloc[train_idx].sum()} ({y.iloc[train_idx].mean()*100:.1f}%)")
print(f"  Test:  1 = {y.iloc[test_idx].sum()} ({y.iloc[test_idx].mean()*100:.1f}%)")
assert y.iloc[test_idx].sum() > 0, "В test нет положительных примеров"
print("В test есть положительные примеры")

Пересечение kepid между train и test: 0
Группы не пересекаются

Баланс классов:
  Train: 1 = 2197 (37.5%)
  Test:  1 = 549 (37.5%)
В test есть положительные примеры


### CV

In [5]:
X_train = X.iloc[train_idx].reset_index(drop=True)
y_train = y.iloc[train_idx].reset_index(drop=True)
groups_train = groups.iloc[train_idx].reset_index(drop=True)

sgkf_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=1)

fold_stats = []
for fold, (tr, va) in enumerate(sgkf_cv.split(X_train, y_train, groups=groups_train)):
    overlap = set(groups_train.iloc[tr]) & set(groups_train.iloc[va])
    n_pos_tr = y_train.iloc[tr].sum()
    n_pos_va = y_train.iloc[va].sum()
    fold_stats.append({
        "fold": fold,
        "train_size": len(tr), "val_size": len(va),
        "train_pos": int(n_pos_tr), "val_pos": int(n_pos_va),
        "overlap": len(overlap),
    })
    assert len(overlap) == 0, f"Утечка в фолде {fold}: {len(overlap)} общих kepid"
    assert n_pos_va > 0, f"В val фолда {fold} нет положительных примеров"

fold_df = pd.DataFrame(fold_stats)
print(fold_df.to_string(index=False))
print("\nВсё хорошо: фолды без утечек, с положительными примерами")

 fold  train_size  val_size  train_pos  val_pos  overlap
    0        4690      1172       1758      439        0
    1        4689      1173       1757      440        0
    2        4689      1173       1757      440        0
    3        4689      1173       1758      439        0
    4        4691      1171       1758      439        0

Всё хорошо: фолды без утечек, с положительными примерами


### Сохранение индексов

In [6]:
# Holdout
pd.DataFrame({"idx": train_idx}).to_csv(FINAL_DIR / "holdout_train_idx.csv", index=False)
pd.DataFrame({"idx": test_idx}).to_csv(FINAL_DIR / "holdout_test_idx.csv", index=False)

# CV-фолды (внутри train)
cv_rows = []
for fold, (tr, va) in enumerate(sgkf_cv.split(X_train, y_train, groups=groups_train)):
    for i in tr:
        cv_rows.append({"fold": fold, "part": "train", "train_local_idx": i})
    for i in va:
        cv_rows.append({"fold": fold, "part": "val", "train_local_idx": i})
pd.DataFrame(cv_rows).to_csv(FINAL_DIR / "cv_folds.csv", index=False)

print("Сохранено:")
print(f"  - holdout_train_idx.csv ({len(train_idx)})")
print(f"  - holdout_test_idx.csv ({len(test_idx)})")
print(f"  - cv_folds.csv ({len(cv_rows)} строк)")

Сохранено:
  - holdout_train_idx.csv (5862)
  - holdout_test_idx.csv (1465)
  - cv_folds.csv (29310 строк)
